In [ ]:
from datasets import load_dataset, Dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
import numpy as np
import evaluate
import matplotlib.pyplot as plt

In [ ]:

dataset = load_dataset("tweet_eval", "sentiment")

extra_train_samples = [
    # -------------------------
    # strong negative
    # -------------------------
    {"text": "The app crashes a lot", "label": 0},
    {"text": "This app keeps crashing", "label": 0},
    {"text": "The application crashes every time", "label": 0},
    {"text": "It crashes again and again", "label": 0},
    {"text": "The app is unstable and crashes often", "label": 0},
    {"text": "The food was awful", "label": 0},
    {"text": "The food tastes awful", "label": 0},
    {"text": "That meal was terrible", "label": 0},
    {"text": "The food was really bad", "label": 0},
    {"text": "Worst food I had today", "label": 0},
    {"text": "This tastes awful", "label": 0},
    {"text": "The meal was awful", "label": 0},
    {"text": "The service was awful", "label": 0},
    {"text": "Awful experience", "label": 0},
    {"text": "The movie was boring", "label": 0},
    {"text": "This movie is boring", "label": 0},
    {"text": "The film was so boring", "label": 0},
    {"text": "What a boring movie", "label": 0},
    {"text": "The movie was dull and boring", "label": 0},
    {"text": "The update broke everything", "label": 0},
    {"text": "This update made things worse", "label": 0},
    {"text": "The service was disappointing", "label": 0},
    {"text": "This is disappointing", "label": 0},
    {"text": "The experience was disappointing", "label": 0},
    {"text": "The product is disappointing", "label": 0},
    {"text": "The phone is disappointing", "label": 0},
    {"text": "The app is terrible", "label": 0},
    {"text": "This service is horrible", "label": 0},
    {"text": "The product is bad", "label": 0},
    {"text": "The movie is bad", "label": 0},
    {"text": "The update is bad", "label": 0},
    {"text": "The service is bad", "label": 0},
    {"text": "This was a bad experience", "label": 0},
    {"text": "The result was terrible", "label": 0},
    {"text": "This is a horrible app", "label": 0},

    # -------------------------
    # strong positive
    # -------------------------
    {"text": "The update improved everything", "label": 2},
    {"text": "This update improved everything", "label": 2},
    {"text": "The new update made it much better", "label": 2},
    {"text": "The update fixed many issues", "label": 2},
    {"text": "Everything works better after the update", "label": 2},
    {"text": "The product works well", "label": 2},
    {"text": "This product works really well", "label": 2},
    {"text": "The product performs well", "label": 2},
    {"text": "It works well so far", "label": 2},
    {"text": "So far the product works great", "label": 2},
    {"text": "This works well", "label": 2},
    {"text": "It works really well", "label": 2},
    {"text": "The app works well now", "label": 2},
    {"text": "Everything works well", "label": 2},
    {"text": "Works as expected", "label": 2},
    {"text": "Pretty decent", "label": 2},
    {"text": "The service was amazing", "label": 2},
    {"text": "This phone is amazing", "label": 2},
    {"text": "I love this product", "label": 2},
    {"text": "I love this phone", "label": 2},
    {"text": "The weather is nice today", "label": 2},
    {"text": "The movie was great", "label": 2},
    {"text": "The experience was excellent", "label": 2},
    {"text": "The product is excellent", "label": 2},
    {"text": "The app is excellent", "label": 2},
    {"text": "This is a great update", "label": 2},
    {"text": "The update is excellent", "label": 2},
    {"text": "This feels much better now", "label": 2},
    {"text": "This is really good", "label": 2},
    {"text": "The product is really good", "label": 2},
    {"text": "The service is really good", "label": 2},
    {"text": "I am very happy with this", "label": 2},
    {"text": "This turned out great", "label": 2},
    {"text": "This is wonderful", "label": 2},
    {"text": "Amazing experience", "label": 2},

    # -------------------------
    # neutral / borderline
    # -------------------------
    {"text": "The movie was okay", "label": 1},
    {"text": "The movie was just okay", "label": 1},
    {"text": "It was okay, nothing special", "label": 1},
    {"text": "The film was fine", "label": 1},
    {"text": "It was average", "label": 1},
    {"text": "It was okay", "label": 1},
    {"text": "The film was just okay", "label": 1},
    {"text": "The movie was average", "label": 1},
    {"text": "It was fine", "label": 1},
    {"text": "Nothing special about the movie", "label": 1},
    {"text": "Could be better", "label": 1},
    {"text": "Nothing impressive", "label": 1},
    {"text": "It is fine", "label": 1},
    {"text": "The meeting was long", "label": 1},
    {"text": "The package arrived yesterday", "label": 1},
    {"text": "The event starts at noon", "label": 1},
    {"text": "The weather is normal today", "label": 1},
    {"text": "The phone is okay", "label": 1},
    {"text": "The product is okay", "label": 1},
    {"text": "The service is okay", "label": 1},
    {"text": "The result is acceptable", "label": 1},
    {"text": "This is acceptable", "label": 1},
    {"text": "This is ordinary", "label": 1},
    {"text": "This feels average", "label": 1},
    {"text": "The movie was neither good nor bad", "label": 1},
    {"text": "The product is neither good nor bad", "label": 1},
    {"text": "The experience was neutral", "label": 1},
    {"text": "It was nothing special", "label": 1},
    {"text": "This is just average", "label": 1},
    {"text": "The update is okay", "label": 1},
    {"text": "It works fine", "label": 1},
    {"text": "The app is fine", "label": 1},
    {"text": "The service is fine", "label": 1},
    {"text": "The movie was decent", "label": 1},
    {"text": "The meal was okay", "label": 1},

    # -------------------------
    # explicit negation: not + negative = positive
    # -------------------------
    {"text": "The phone is not bad", "label": 2},
    {"text": "The movie is not bad", "label": 2},
    {"text": "This product is not bad", "label": 2},
    {"text": "Not bad at all", "label": 2},
    {"text": "not bad", "label": 2},
    {"text": "this phone is not bad", "label": 2},
    {"text": "This update is not bad", "label": 2},
    {"text": "The service is not bad", "label": 2},
    {"text": "The app is not bad", "label": 2},
    {"text": "That was not bad", "label": 2},
    {"text": "The movie was not bad", "label": 2},
    {"text": "The meal was not bad", "label": 2},
    {"text": "The result is not bad", "label": 2},
    {"text": "It is not bad", "label": 2},
    {"text": "Actually not bad", "label": 2},

    # -------------------------
    # explicit negation: not + positive = negative
    # -------------------------
    {"text": "The movie is not good", "label": 0},
    {"text": "The service is not good", "label": 0},
    {"text": "The update is not good", "label": 0},
    {"text": "not good", "label": 0},
    {"text": "this movie is not good", "label": 0},
    {"text": "the service is not good", "label": 0},
    {"text": "The update is not great", "label": 0},
    {"text": "The phone is not great", "label": 0},
    {"text": "not great", "label": 0},
    {"text": "The movie was not good", "label": 0},
    {"text": "The experience was not good", "label": 0},
    {"text": "This was not good", "label": 0},
    {"text": "This is not good at all", "label": 0},
    {"text": "The product is not good", "label": 0},
    {"text": "The app is not great", "label": 0},
    {"text": "The result is not great", "label": 0},
    {"text": "That was not great", "label": 0},
    {"text": "The service was not great", "label": 0},
    {"text": "This is not amazing", "label": 0},
    {"text": "The phone is not amazing", "label": 0},

    # -------------------------
    # explicit negation: not terrible / not awful
    # -------------------------
    {"text": "Not terrible", "label": 2},
    {"text": "not terrible", "label": 2},
    {"text": "this is not terrible", "label": 2},
    {"text": "The movie is not terrible", "label": 2},
    {"text": "The service is not terrible", "label": 2},
    {"text": "The food is not terrible", "label": 2},
    {"text": "The app is not terrible", "label": 2},
    {"text": "Not awful", "label": 2},
    {"text": "The movie is not awful", "label": 2},
    {"text": "The service is not awful", "label": 2},
    {"text": "The food is not awful", "label": 2},

    # -------------------------
    # mixed / contrastive neutral
    # -------------------------
    {"text": "Not great, not terrible", "label": 1},
    {"text": "Not good, not bad", "label": 1},
    {"text": "Neither good nor bad", "label": 1},
    {"text": "It is not amazing but not awful", "label": 1},
    {"text": "The movie is not great but not bad", "label": 1},
    {"text": "The service is not good but not terrible", "label": 1},
    {"text": "The app is not bad but not impressive", "label": 1},
    {"text": "The product is not great but okay", "label": 1},
    {"text": "It is okay, not great", "label": 1},
    {"text": "It is fine, not amazing", "label": 1},

    # -------------------------
    # implicit negative
    # -------------------------
    {"text": "I expected better", "label": 0},
    {"text": "I expected more", "label": 0},
    {"text": "I was expecting better", "label": 0},
    {"text": "I thought it would be better", "label": 0},
    {"text": "This could have been better", "label": 0},
    {"text": "This should have been better", "label": 0},
    {"text": "I am not impressed", "label": 0},
    {"text": "This is not impressive", "label": 0},
    {"text": "That was underwhelming", "label": 0},
    {"text": "Very underwhelming", "label": 0},
    {"text": "This is frustrating", "label": 0},
    {"text": "This is annoying", "label": 0},
    {"text": "That was a letdown", "label": 0},
    {"text": "This was a letdown", "label": 0},
    {"text": "I am disappointed", "label": 0},
    {"text": "Still disappointing", "label": 0},
    {"text": "The result is underwhelming", "label": 0},
    {"text": "This feels underwhelming", "label": 0},
    {"text": "I wanted more from this", "label": 0},
    {"text": "This is worse than expected", "label": 0},

    # -------------------------
    # implicit positive
    # -------------------------
    {"text": "Better than expected", "label": 2},
    {"text": "Much better than expected", "label": 2},
    {"text": "Surprisingly good", "label": 2},
    {"text": "This is impressive", "label": 2},
    {"text": "I am impressed", "label": 2},
    {"text": "This turned out better than expected", "label": 2},
    {"text": "The result is impressive", "label": 2},
    {"text": "This exceeded expectations", "label": 2},
    {"text": "Way better than before", "label": 2},
    {"text": "This is satisfying", "label": 2},
    {"text": "Very satisfying", "label": 2},
    {"text": "This feels great now", "label": 2},
    {"text": "This is much improved", "label": 2},
    {"text": "Huge improvement", "label": 2},
    {"text": "A solid improvement", "label": 2},

    # -------------------------
    # ambiguous / weak neutral
    # -------------------------
    {"text": "It could be worse", "label": 1},
    {"text": "Could be worse", "label": 1},
    {"text": "At least it works", "label": 1},
    {"text": "It is what it is", "label": 1},
    {"text": "I have seen worse", "label": 1},
    {"text": "Not the best", "label": 1},
    {"text": "Not the worst either", "label": 1},
    {"text": "It does the job", "label": 1},
    {"text": "Good enough", "label": 1},
    {"text": "Just okay overall", "label": 1},
    {"text": "Nothing to complain about", "label": 1},
    {"text": "Nothing to praise either", "label": 1},
    {"text": "It is passable", "label": 1},
    {"text": "This is workable", "label": 1},
    {"text": "That is fair enough", "label": 1},

    # -------------------------
    # weak positive
    # -------------------------
    {"text": "Fairly good", "label": 2},
    {"text": "Quite good", "label": 2},
    {"text": "Good overall", "label": 2},
    {"text": "This is decent", "label": 2},
    {"text": "A decent product", "label": 2},
    {"text": "A decent movie", "label": 2},
    {"text": "The service was decent", "label": 2},
    {"text": "The app is decent", "label": 2},
    {"text": "Pretty good", "label": 2},
    {"text": "Actually pretty good", "label": 2},
    {"text": "More than okay", "label": 2},
    {"text": "Better now", "label": 2},
    {"text": "This is better now", "label": 2},
    {"text": "This feels better", "label": 2},
    {"text": "I am satisfied", "label": 2},

    # -------------------------
    # weak negative
    # -------------------------
    {"text": "Not very good", "label": 0},
    {"text": "Not that good", "label": 0},
    {"text": "Could be improved", "label": 0},
    {"text": "Needs improvement", "label": 0},
    {"text": "This needs work", "label": 0},
    {"text": "Still not good enough", "label": 0},
    {"text": "This is below average", "label": 0},
    {"text": "Slightly disappointing", "label": 0},
    {"text": "A bit disappointing", "label": 0},
    {"text": "Not really worth it", "label": 0},
    {"text": "This is lacking", "label": 0},
    {"text": "This feels weak", "label": 0},
    {"text": "This is poor", "label": 0},
    {"text": "The result is poor", "label": 0},
    {"text": "That was poor", "label": 0},

    # -------------------------
    # contrast pairs
    # -------------------------
    {"text": "The movie is good", "label": 2},
    {"text": "The movie is not good", "label": 0},
    {"text": "The service is great", "label": 2},
    {"text": "The service is not great", "label": 0},
    {"text": "The phone is bad", "label": 0},
    {"text": "The phone is not bad", "label": 2},
    {"text": "The food is awful", "label": 0},
    {"text": "The food is not awful", "label": 2},
    {"text": "The app is terrible", "label": 0},
    {"text": "The app is not terrible", "label": 2},
    {"text": "The update is great", "label": 2},
    {"text": "The update is not great", "label": 0},
    {"text": "The product is okay", "label": 1},
    {"text": "The product is excellent", "label": 2},
    {"text": "The product is disappointing", "label": 0},
]

seen = set()
cleaned_samples = []

for item in extra_train_samples:
    key = (item["text"].strip().lower(), item["label"])
    if key not in seen:
        seen.add(key)
        cleaned_samples.append(item)

extra_train_samples = cleaned_samples

extra_dataset = Dataset.from_list(
    extra_train_samples,
    features=dataset["train"].features
)

new_train = concatenate_datasets([dataset["train"], extra_dataset])

dataset["train"] = new_train

print(dataset)

In [ ]:
model_id = "model"
sentiment_model_id = "classifier"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=3,
    ignore_mismatched_sizes=True,
)

model_size = sum(t.numel() for t in model.parameters())
print(f"Transformer size: {model_size/1000**2:.1f}M parameters")

label_names = dataset["train"].features["label"].names
model.config.id2label = {i: name for i, name in enumerate(label_names)}
model.config.label2id = {name: i for i, name in enumerate(label_names)}

In [ ]:
def tokenize(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=64,
    )

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.remove_columns(["text"])
dataset = dataset.rename_column("label", "labels")
dataset.set_format("torch")
dataset

In [ ]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy.compute(predictions=preds, references=labels)["accuracy"]
    f1_macro = f1.compute(
        predictions=preds,
        references=labels,
        average="macro"
    )["f1"]

    return {
        "accuracy": acc,
        "f1_macro": f1_macro,
    }

training_args = TrainingArguments(
    output_dir=sentiment_model_id,
    learning_rate=2e-5,
    weight_decay=0.02,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    eval_strategy="steps",
    save_strategy="steps",
    logging_strategy="steps",
    logging_steps=500,
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none",
    fp16=True,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model(sentiment_model_id)

In [ ]:
print("Best checkpoint:", trainer.state.best_model_checkpoint)

In [ ]:
logs = trainer.state.log_history

train_steps = [log["step"] for log in logs if "loss" in log and "eval_loss" not in log]
train_losses = [log["loss"] for log in logs if "loss" in log and "eval_loss" not in log]

eval_steps = [log["step"] for log in logs if "eval_loss" in log]
eval_losses = [log["eval_loss"] for log in logs if "eval_loss" in log]

plt.figure(figsize=(12, 6))
plt.plot(train_steps, train_losses, label="Training Loss", linewidth=2)
plt.plot(eval_steps, eval_losses, label="Validation Loss", linewidth=2)
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.3)
plt.show()

In [ ]:
test_metrics = trainer.evaluate(dataset["test"])
print(test_metrics)

In [ ]:
from transformers import pipeline

classifier = pipeline(
    task="text-classification",
    model=sentiment_model_id,
    tokenizer=sentiment_model_id,
)

print(classifier("I love this phone"))
print(classifier("This is the worst service ever"))